In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "src"), str(ROOT / "setup")]

print(ROOT)

/Users/prasunbanerjee/Desktop/CommodityResearch


In [23]:
import pandas as pd
import numpy as np 
import polars as pl
import xarray as xr
import boto3
import eccodes

from dotenv import load_dotenv
from setup.paths import DATA_DIR
from infra.pipeline.storage.s3_client import S3Client
from tempfile import TemporaryFile
from comm_research.markets.weather.map import ZONAL_MAP

In [24]:
load_dotenv(ROOT / "infra" / ".env.garage")

True

In [25]:
BUCKET = 'commodities-raw'
KEY = 'ecmwf/source=cds/system=51/product=seasonal-monthly-mean/param=2t/date=20261101/fcmonth=01/forecast.grib'

In [26]:
def show_grib2(raw_data: bytes) -> pd.DataFrame:
    """Return one row per grid point in each GRIB message."""

    keys = [
        "shortName", "units", "typeOfLevel", "level", "dataDate", "dataTime",
        "stepRange", "validityDate", "validityTime", "perturbationNumber",
    ]
    frames = []
    try:
        with TemporaryFile() as grib:
            grib.write(raw_data)
            grib.seek(0)
            while (message := eccodes.codes_grib_new_from_file(grib)) is not None:
                try:
                    points = eccodes.codes_get_array(message, "latLonValues").reshape(-1, 3)
                    if eccodes.codes_get(message, "bitmapPresent"):
                        bitmap = eccodes.codes_get_array(message, "bitmap")
                        points[bitmap == 0, 2] = float("nan")

                    frame = pd.DataFrame(points, columns=["latitude", "longitude", "value"])
                    frame["message"] = len(frames)
                    for key in keys:
                        if eccodes.codes_is_defined(message, key) and not eccodes.codes_is_missing(message, key):
                            frame[key] = eccodes.codes_get(message, key)
                    frames.append(frame)
                finally:
                    eccodes.codes_release(message)
    except eccodes.CodesInternalError as error:
        raise ValueError(f"Could not decode raw_data as GRIB: {error}") from error

    if not frames:
        raise ValueError("No GRIB messages found in raw_data")

    return pd.concat(frames, ignore_index=True)

In [27]:
s3 = S3Client()

In [28]:
raw_data = s3.read_from_lake(BUCKET, KEY)
df = show_grib2(raw_data)

In [29]:
min_lat, max_lat, min_lon, max_lon = ZONAL_MAP["us48"]
lon = (df["longitude"] + 180) % 360 - 180
us48 = df[df["latitude"].between(min_lat, max_lat) & lon.between(min_lon, max_lon)]

In [ ]:
us48.to_parquet()

,latitude,longitude,value,message,shortName,units,typeOfLevel,level,dataDate,dataTime,stepRange,validityDate,validityTime,perturbationNumber
14635,49.5,235.5,279.784210,0,2t,K,surface,0,20261101,0,720,20261201,0,0
14636,49.5,236.5,278.120148,0,2t,K,surface,0,20261101,0,720,20261201,0,0
14637,49.5,237.5,275.383820,0,2t,K,surface,0,20261101,0,720,20261201,0,0
14638,49.5,238.5,273.012726,0,2t,K,surface,0,20261101,0,720,20261201,0,0
14639,49.5,239.5,271.743195,0,2t,K,surface,0,20261101,0,720,20261201,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3263689,24.5,289.5,299.200745,50,2t,K,surface,0,20261101,0,720,20261201,0,50
3263690,24.5,290.5,299.222229,50,2t,K,surface,0,20261101,0,720,20261201,0,50
3263691,24.5,291.5,299.233948,50,2t,K,surface,0,20261101,0,720,20261201,0,50
3263692,24.5,292.5,299.235901,50,2t,K,surface,0,20261101,0,720,20261201,0,50
